<a href="https://colab.research.google.com/github/nikolas-joyce/macro-signal-strategies/blob/main/04-credit-cycle/notebooks/02_signal_construction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Cycle - Notebook 2: Signal Construction


## 1. Load Aligned Panel


In [ ]:
# Cell 1: Drive mount (top of every notebook)
from google.colab import drive
drive.mount('/content/drive')
BASE_PATH = '/content/drive/MyDrive/macro_strategies/'
import os
os.makedirs(BASE_PATH + 'data/raw', exist_ok=True)
os.makedirs(BASE_PATH + 'data/processed', exist_ok=True)


In [ ]:
import pandas as pd
import numpy as np
from src.signal_builder import normalise_signal
panel = pd.read_parquet(BASE_PATH + 'data/processed/aligned_panel.parquet')


## 2. Raw Signal Computation

Implement the strategy's core formula (see `index.md`).


In [ ]:
# Excess credit growth: total loan growth minus GDP growth (quarterly YoY)
credit_g = panel['TOTLL'].pct_change(4) * 100
gdp_g    = panel['GDPC1'].pct_change(4) * 100
raw = credit_g - gdp_g

## 3. Signal Transformations

Rolling z-score, +/- 3 sigma clip, lag-1.


In [ ]:
signal = normalise_signal(raw, window=60)


## 4. Alternative Signal Variants

Headline vs. core, 3m vs. 6m, level vs. change.


In [ ]:
credit_g   = panel['TOTLL'].pct_change(4) * 100
busloans_g = panel['BUSLOANS'].pct_change(4) * 100 if 'BUSLOANS' in panel.columns else credit_g
gdp_g      = panel['GDPC1'].pct_change(4) * 100
variants = pd.DataFrame({
    'totll_vs_gdp':    normalise_signal(credit_g - gdp_g),
    'busloans_vs_gdp': normalise_signal(busloans_g - gdp_g),
}).dropna(axis=1, how='all')
variants.plot(figsize=(12, 5), title='Credit Cycle Signal Variants')

## 5. Signal Validation

Time-series plot, distribution histogram, autocorrelation.


In [ ]:
signal.plot(figsize=(12, 5), title='Signal (z-score)')


## 6. Predictive Power Pre-Backtest

Rolling Information Coefficient, quantile forward returns.


In [ ]:
import matplotlib.pyplot as plt

mkt_returns = pd.read_parquet(BASE_PATH + 'data/processed/market_returns.parquet')
fwd_returns = mkt_returns.iloc[:, 0].shift(-1)

aligned = pd.concat([signal, fwd_returns], axis=1).dropna()
aligned.columns = ['signal', 'fwd_ret']

ic = aligned['signal'].rolling(12).corr(aligned['fwd_ret'])
fig, ax = plt.subplots(figsize=(12, 4))
ic.plot(ax=ax, title='Rolling 12m Information Coefficient (vs primary asset fwd return)')
ax.axhline(0, color='black', linewidth=0.5, alpha=0.5)
ax.set_ylabel('IC (Pearson)')
plt.tight_layout()
plt.show()

aligned['quintile'] = pd.qcut(aligned['signal'], 5, labels=False, duplicates='drop')
qmeans = aligned.groupby('quintile')['fwd_ret'].mean() * 12
fig2, ax2 = plt.subplots(figsize=(8, 4))
qmeans.plot(kind='bar', ax=ax2, color='steelblue',
            title='Signal Quintile — Mean Annualised Forward Return')
ax2.set_xlabel('Quintile (1=low, 5=high)')
ax2.set_ylabel('Annualised Return')
plt.tight_layout()
plt.show()

## 7. Export


In [ ]:
signal.to_frame('signal').to_parquet(
    BASE_PATH + 'data/processed/signals.parquet'
)


## Limitations

- Rolling z-score is symmetric -- asymmetric distributions may bias.
- Lag of 1 assumes monthly release cadence.
